##### The following practice code is intended for educational purposes only. For contact :  audit@korea.ac.kr, Sungryel Lim Ph.D

##### This practice code is not a completed commercial version but has been developed for educational purposes.

# PEFT(LoRA) 적용 전/후 파라미터 비교

이 Notebook은 본격적인 Fine-Tuning 전에 **LoRA를 적용하면 어떤 파라미터가 학습 대상이 되는지** 확인하는 간단한 예제입니다.

이번 실습에서는 Base Model을 프로젝트 내부에 별도로 복사하지 않습니다.

`Qwen/Qwen2.5-1.5B-Instruct`를 `from_pretrained()`로 불러오며, Hugging Face가 관리하는 **로컬 캐시를 자동으로 재사용**합니다.

- 이미 다운로드되어 있으면: 기존 캐시 사용
- 다운로드되어 있지 않으면: 최초 1회 다운로드
- 이후 `.py` 코드에서도 동일한 모델 ID를 사용하면 같은 캐시를 재사용할 수 있습니다.


## 1. 필요한 라이브러리 불러오기

`transformers`는 Base Model 로딩에 사용하고, `peft`는 LoRA를 적용하는 데 사용합니다.


In [6]:
# Jupyter Notebook의 tqdm 진행률 표시를 위해 추가 설치
%pip install ipywidgets


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [7]:
from pathlib import Path

from transformers import AutoModelForCausalLM
from peft import LoraConfig, get_peft_model


## 2. Hugging Face 캐시 위치 확인

Hugging Face Hub는 다운로드한 모델을 기본적으로 사용자 캐시 디렉토리에 저장합니다.

아래 셀에서는 **현재 환경에서 실제로 사용되는 Hugging Face Hub 캐시 경로**를 확인합니다.

> 참고: `HF_HOME` 또는 `HF_HUB_CACHE` 환경 변수를 설정한 경우 기본 위치와 달라질 수 있습니다.


In [8]:
import os
from huggingface_hub.constants import HF_HUB_CACHE

print("현재 작업 디렉토리(CWD)")
print(" →", os.getcwd())

print("\nHugging Face Hub 캐시 경로")
print(" →", HF_HUB_CACHE)

print("\n캐시 디렉토리 존재 여부")
print(" →", Path(HF_HUB_CACHE).exists())


현재 작업 디렉토리(CWD)
 → /Users/seohyeokin/workspace/skala-sLLM/sllm-main-std

Hugging Face Hub 캐시 경로
 → /Users/seohyeokin/.cache/huggingface/hub

캐시 디렉토리 존재 여부
 → True


## 3. Qwen Base Model의 실제 캐시 Snapshot 위치 확인

Hugging Face 캐시 내부에는 모델이 revision(snapshot) 단위로 저장됩니다.

아래 코드는 **이미 캐시에 존재하는 Qwen2.5-1.5B-Instruct의 실제 로컬 경로**를 찾아봅니다.

아직 모델을 다운로드하지 않은 환경이라면 `캐시에서 아직 찾을 수 없습니다.`라고 표시될 수 있습니다.


In [9]:
from huggingface_hub import try_to_load_from_cache

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

# config.json의 실제 캐시 위치를 이용해
# 현재 사용 중인 모델 snapshot 디렉토리를 확인합니다.
cached_config = try_to_load_from_cache(
    repo_id=MODEL_NAME,
    filename="config.json"
)

if isinstance(cached_config, str):
    snapshot_path = Path(cached_config).parent

    print("Qwen Base Model 캐시 Snapshot")
    print(" →", snapshot_path)
else:
    print("Qwen Base Model을 캐시에서 아직 찾을 수 없습니다.")
    print("다음 셀에서 모델을 로드하면 필요한 파일이 다운로드됩니다.")


Qwen Base Model 캐시 Snapshot
 → /Users/seohyeokin/.cache/huggingface/hub/models--Qwen--Qwen2.5-1.5B-Instruct/snapshots/989aa7980e4cf806f80c7fef2b1adb7bc71aa306


## 4. Qwen Base Model 로드

기존 sLLM 코드와 동일하게 Hugging Face 모델 ID를 사용합니다.

```python
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
```

`from_pretrained()`는 캐시에 모델이 있으면 **다시 다운로드하지 않고 기존 파일을 재사용**합니다.


In [10]:
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME
)

print("Base Model 로드 완료")


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Base Model 로드 완료


## 5. 모델 로드 후 실제 캐시 위치 다시 확인

최초 실행 환경이었다면 바로 앞 셀에서 모델이 다운로드되었을 수 있습니다.

따라서 모델 로드가 끝난 뒤 다시 실제 snapshot 경로를 확인합니다.


In [11]:
cached_config = try_to_load_from_cache(
    repo_id=MODEL_NAME,
    filename="config.json"
)

if isinstance(cached_config, str):
    snapshot_path = Path(cached_config).parent

    print("현재 사용 중인 Qwen Base Model의 로컬 캐시 위치")
    print(" →", snapshot_path)
else:
    print("캐시 위치를 확인하지 못했습니다.")


현재 사용 중인 Qwen Base Model의 로컬 캐시 위치
 → /Users/seohyeokin/.cache/huggingface/hub/models--Qwen--Qwen2.5-1.5B-Instruct/snapshots/989aa7980e4cf806f80c7fef2b1adb7bc71aa306


## 6. LoRA 적용 전 파라미터 확인

현재는 일반 Base Model이므로 기본적으로 전체 모델 파라미터가 학습 가능한 상태입니다.


In [12]:
# 모델의 전체 파라미터 수와 실제 학습 가능한 파라미터 수를 계산하는 함수
def count_parameters(model):

    # model.parameters()
    # → 모델이 가지고 있는 모든 파라미터(weight, bias 등)를 하나씩 가져옵니다.
    #
    # p.numel()
    # → 해당 파라미터에 포함된 숫자의 개수를 반환합니다.
    #   예: (100, 200) 크기의 가중치라면 100 × 200 = 20,000개
    #
    # 따라서 모든 파라미터의 numel()을 더하면
    # 모델 전체의 파라미터 개수를 구할 수 있습니다.
    total_params = sum(
        p.numel()
        for p in model.parameters()
    )

    # 실제 학습되는 파라미터의 개수만 계산합니다.
    #
    # p.requires_grad == True
    # → 역전파(Backpropagation)를 통해 값이 업데이트되는 파라미터
    #
    # LoRA 적용 후에는 기존 Base Model의 대부분이 False(Freeze)가 되고,
    # LoRA Adapter의 파라미터만 True가 됩니다.
    trainable_params = sum(
        p.numel()
        for p in model.parameters()
        if p.requires_grad
    )

    # 전체 파라미터 수와 학습 가능한 파라미터 수를 반환
    return total_params, trainable_params


total_before, trainable_before = count_parameters(model)

print("===== LoRA 적용 전 =====")
print(f"전체 파라미터      : {total_before:,}")
print(f"학습 가능 파라미터 : {trainable_before:,}")
print(f"학습 가능 비율     : {trainable_before / total_before * 100:.4f}%")


===== LoRA 적용 전 =====
전체 파라미터      : 1,543,714,304
학습 가능 파라미터 : 1,543,714,304
학습 가능 비율     : 100.0000%


## 7. LoRA 설정

이번 예제에서는 Attention의 `q_proj`, `v_proj`에 LoRA Adapter를 적용합니다.

- `r=8`: LoRA의 rank
- `lora_alpha=16`: LoRA scaling 값
- `target_modules`: LoRA를 적용할 레이어
- `lora_dropout=0.05`: LoRA 경로에 적용할 dropout

이 단계에서는 실제 Fine-Tuning을 수행하지 않습니다. **LoRA 구조만 모델에 추가**합니다.


In [13]:
# LoRA(Low-Rank Adaptation)의 학습 방식을 설정합니다.
#
# LoRA는 기존 Base Model의 가중치를 직접 수정하는 대신,
# 일부 레이어에 작은 크기의 학습용 행렬(LoRA Adapter)을 추가합니다.
lora_config = LoraConfig(

    # LoRA에서 추가할 행렬의 Rank(차원)를 지정합니다.
    #
    # r이 작을수록:
    # -> 학습해야 하는 파라미터 수가 적어지고 학습이 가벼워집니다.
    #
    # r이 클수록:
    # -> 더 많은 정보를 학습할 수 있지만 파라미터와 메모리 사용량도 증가합니다.
    #
    # 일반적으로 8, 16, 32 등의 값을 많이 사용합니다.
    r=8,

    # LoRA가 학습한 변화량을 얼마나 강하게 반영할지 조절하는 값입니다.
    #
    # 쉽게 말하면 LoRA Adapter의 영향력을 조절하는 값입니다.
    # 보통 r과 함께 설정하며, 여기서는 r=8, alpha=16으로 설정합니다.
    lora_alpha=16,

    # LoRA Adapter를 추가할 레이어를 지정합니다.
    #
    # Qwen의 Self-Attention에는 q_proj, k_proj, v_proj, o_proj 등이 있는데,
    # 이번 간단한 실습에서는 q_proj와 v_proj에만 LoRA를 적용합니다.
    #
    # q_proj : Query를 만드는 레이어
    # v_proj : Value를 만드는 레이어
    target_modules=[
        "q_proj",
        "v_proj"
    ],

    # LoRA 학습 과정에서 일부 값을 확률적으로 제외(Dropout)합니다.
    #
    # 0.05는 5%의 Dropout을 적용한다는 의미이며,
    # 학습 데이터에 지나치게 맞춰지는 과적합(Overfitting)을
    # 줄이는 데 도움을 줄 수 있습니다.
    lora_dropout=0.05,

    # 기존 모델의 bias를 추가로 학습할지 결정합니다.
    #
    # "none"은 기존 bias를 학습하지 않는다는 의미입니다.
    # 따라서 이번 실습에서는 LoRA Adapter 파라미터에 집중합니다.
    bias="none",

    # 어떤 종류의 모델에 LoRA를 적용하는지 지정합니다.
    #
    # CAUSAL_LM = 이전에 등장한 Token들을 바탕으로
    # 다음 Token을 예측하는 언어 모델
    #
    # Qwen2.5-Instruct와 같은 생성형 LLM이 여기에 해당합니다.
    task_type="CAUSAL_LM"
)

model = get_peft_model(
    model,
    lora_config
)

print("LoRA 적용 완료")


LoRA 적용 완료


## 8. LoRA 적용 후 파라미터 비교

LoRA를 적용하면 Base Model의 기존 파라미터 대부분은 Freeze되고, 새롭게 추가된 LoRA Adapter 파라미터만 학습 대상이 됩니다.

중요한 점은 **전체 모델의 파라미터 자체가 작아지는 것이 아니라, 실제 학습해야 하는 파라미터가 크게 줄어든다**는 것입니다.


In [14]:
total_after, trainable_after = count_parameters(model)

print("===== LoRA 적용 후 =====")
print(f"전체 파라미터      : {total_after:,}")
print(f"학습 가능 파라미터 : {trainable_after:,}")
print(f"학습 가능 비율     : {trainable_after / total_after * 100:.4f}%")

print("\n===== PEFT 공식 출력 =====")
model.print_trainable_parameters()


===== LoRA 적용 후 =====
전체 파라미터      : 1,544,803,840
학습 가능 파라미터 : 1,089,536
학습 가능 비율     : 0.0705%

===== PEFT 공식 출력 =====
trainable params: 1,089,536 || all params: 1,544,803,840 || trainable%: 0.0705


## 9. 실제 학습 대상 파라미터 확인

`requires_grad=True`인 파라미터만 출력합니다.

출력 결과에서 `lora_A`, `lora_B`가 포함된 파라미터들을 확인해 보세요.


In [15]:
print("===== 실제 학습되는 파라미터 =====")

# 모델에서 실제로 학습되는 파라미터만 찾아서 출력합니다.
#
# model.named_parameters()
# → 모델이 가지고 있는 모든 파라미터를 하나씩 가져옵니다.
# → 각 파라미터의 이름(name)과 실제 값(param)을 함께 반환합니다.
#
# 예:
# name  = "...q_proj.lora_A.default.weight"
# param = 해당 가중치가 저장된 Tensor
for name, param in model.named_parameters():

    # requires_grad가 True인 파라미터만 확인합니다.
    #
    # requires_grad = True
    # → 학습 과정에서 역전파(Backpropagation)를 통해 값이 변경됨
    #
    # requires_grad = False
    # → 학습하지 않는 파라미터(Freeze된 파라미터)
    #
    # LoRA 적용 후에는 Base Model의 기존 파라미터는 대부분 False가 되고,
    # 새롭게 추가된 LoRA Adapter의 파라미터가 True가 됩니다.
    if param.requires_grad:

        # 학습되는 파라미터의 정보를 출력합니다.
        print(

            # 파라미터의 이름을 출력합니다.
            #
            # <90은 출력 영역을 90칸으로 확보하고 왼쪽 정렬하여
            # 여러 파라미터를 출력했을 때 보기 좋게 맞추기 위한 설정입니다.
            f"{name:<90} "

            # 해당 파라미터 Tensor의 모양(Shape)을 출력합니다.
            #
            # 예: [8, 1536]
            # → 8 × 1536 형태의 2차원 Tensor라는 의미입니다.
            f"shape={str(list(param.shape)):<18} "

            # 해당 Tensor 안에 들어 있는 전체 숫자의 개수를 출력합니다.
            #
            # param.numel()
            # → Number of Elements
            #
            # 예: shape=[8, 1536]
            # → 8 × 1536 = 12,288개의 파라미터
            #
            # :, 옵션은 12,288처럼 천 단위 쉼표를 표시합니다.
            f"params={param.numel():,}"
        )


===== 실제 학습되는 파라미터 =====
base_model.model.model.layers.0.self_attn.q_proj.lora_A.default.weight                     shape=[8, 1536]          params=12,288
base_model.model.model.layers.0.self_attn.q_proj.lora_B.default.weight                     shape=[1536, 8]          params=12,288
base_model.model.model.layers.0.self_attn.v_proj.lora_A.default.weight                     shape=[8, 1536]          params=12,288
base_model.model.model.layers.0.self_attn.v_proj.lora_B.default.weight                     shape=[256, 8]           params=2,048
base_model.model.model.layers.1.self_attn.q_proj.lora_A.default.weight                     shape=[8, 1536]          params=12,288
base_model.model.model.layers.1.self_attn.q_proj.lora_B.default.weight                     shape=[1536, 8]          params=12,288
base_model.model.model.layers.1.self_attn.v_proj.lora_A.default.weight                     shape=[8, 1536]          params=12,288
base_model.model.model.layers.1.self_attn.v_proj.lora_B.default.we

## 정리

이번 실습에서 확인한 핵심 내용은 다음과 같습니다.

1. Base Model은 `Qwen/Qwen2.5-1.5B-Instruct`를 사용합니다.
2. Base Model 파일은 Hugging Face의 로컬 캐시에서 자동으로 관리됩니다.
3. 이미 다운로드된 모델은 이후 Notebook이나 `.py`에서 동일한 모델 ID를 사용하면 재사용됩니다.
4. LoRA는 Base Model 전체를 다시 학습하지 않고 작은 Adapter 파라미터(`lora_A`, `lora_B`)를 추가해 학습합니다.
5. 따라서 전체 파라미터 수는 거의 그대로이지만 **Trainable Parameter 비율은 크게 감소**합니다.

다음 Fine-Tuning 실습에서도 같은 `MODEL_NAME`을 사용하면 동일한 Base Model 캐시를 그대로 활용할 수 있습니다.
